# Lab01S01 — Consulta GraphQL aos 100 repositórios mais populares do GitHub

| RQ | Pergunta | Métrica |
|:--:|----------|--------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até última atualização |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e updates por grupo de linguagem |

## 1. Setup

In [ ]:
import requests
import json
import time
from datetime import datetime, timezone
import os
import pandas as pd

#paste here your github token
GITHUB_TOKEN = ""
GITHUB_API = "https://api.github.com/graphql"
HEADERS = {"Authorization": f"Bearer {GITHUB_TOKEN}"}

assert GITHUB_TOKEN, "Defina seu GITHUB_TOKEN antes de continuar!"
print("Token OK")

Token OK


## 2. Query GraphQL

In [2]:
QUERY = """
{
  search(query: "stars:>1", type: REPOSITORY, first: 10, after: CURSOR) {
    pageInfo { hasNextPage endCursor }
    nodes {
      ... on Repository {
        nameWithOwner
        stargazerCount
        createdAt
        updatedAt
        primaryLanguage { name }
        mergedPRs: pullRequests(states: MERGED) { totalCount }
        releases { totalCount }
        allIssues: issues { totalCount }
        closedIssues: issues(states: CLOSED) { totalCount }
      }
    }
  }
}
"""

## 3. Requisição automática (com retry)

In [3]:
MAX_RETRIES = 5

def fetch_graphql(query_str):
    for attempt in range(1, MAX_RETRIES + 1):
        r = requests.post(GITHUB_API, json={"query": query_str}, headers=HEADERS)
        if r.status_code == 200:
            data = r.json()
            if "errors" in data:
                raise RuntimeError(f"GraphQL errors: {data['errors']}")
            return data
        if r.status_code in (502, 503, 504) and attempt < MAX_RETRIES:
            wait = 2 ** attempt
            print(f"  [retry {attempt}] HTTP {r.status_code} — aguardando {wait}s...")
            time.sleep(wait)
        else:
            r.raise_for_status()

## 4. Coleta dos 100 repositórios

In [4]:
TARGET = 100
nodes_raw = []
cursor = None

while len(nodes_raw) < TARGET:
    q = QUERY.replace("after: CURSOR", f'after: \"{cursor}\"' if cursor else "")
    result = fetch_graphql(q)
    page = result["data"]["search"]
    nodes_raw.extend(page["nodes"])
    print(f"  {len(nodes_raw)}/{TARGET} coletados")

    if not page["pageInfo"]["hasNextPage"]:
        break
    cursor = page["pageInfo"]["endCursor"]
    time.sleep(3)

nodes_raw = nodes_raw[:TARGET]
print(f"\nColeta finalizada: {len(nodes_raw)} repositórios")

  10/100 coletados
  20/100 coletados
  30/100 coletados
  40/100 coletados
  50/100 coletados
  60/100 coletados
  70/100 coletados
  80/100 coletados
  90/100 coletados
  100/100 coletados

Coleta finalizada: 100 repositórios


## 5. Montar DataFrame com pandas

In [5]:
now = datetime.now(timezone.utc)

rows = []
for r in nodes_raw:
    created = pd.to_datetime(r["createdAt"])
    updated = pd.to_datetime(r["updatedAt"])
    total = r["allIssues"]["totalCount"]
    closed = r["closedIssues"]["totalCount"]

    rows.append({
        "nome":              r["nameWithOwner"],
        "estrelas":          r["stargazerCount"],
        "criado_em":         created,
        "idade_dias":        (now - created).days,
        "atualizado_em":     updated,
        "dias_desde_update": (now - updated).days,
        "linguagem":         r["primaryLanguage"]["name"] if r["primaryLanguage"] else None,
        "prs_aceitas":       r["mergedPRs"]["totalCount"],
        "releases":          r["releases"]["totalCount"],
        "total_issues":      total,
        "issues_fechadas":   closed,
        "razao_fechadas":    round(closed / total, 4) if total > 0 else 0.0,
    })

df = pd.DataFrame(rows)
df.index += 1
df.index.name = "#"

print(f"DataFrame criado: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(10)

DataFrame criado: 100 linhas x 12 colunas


,nome,estrelas,criado_em,idade_dias,atualizado_em,dias_desde_update,linguagem,prs_aceitas,releases,total_issues,issues_fechadas,razao_fechadas
#,,,,,,,,,,,,
1,codecrafters-io/build-your-own-x,472636,2018-05-09 12:03:18+00:00,2857,2026-03-06 05:17:30+00:00,0,Markdown,155,0,888,653,0.7354
2,sindresorhus/awesome,442933,2014-07-11 13:42:37+00:00,4255,2026-03-06 05:11:24+00:00,0,None,694,0,365,349,0.9562
3,freeCodeCamp/freeCodeCamp,437813,2014-12-24 17:49:19+00:00,4089,2026-03-06 04:07:06+00:00,0,TypeScript,27772,0,21154,20978,0.9917
4,public-apis/public-apis,404586,2016-03-20 23:49:42+00:00,3637,2026-03-06 04:57:12+00:00,0,Python,1872,0,839,833,0.9928
5,EbookFoundation/free-programming-books,383644,2013-10-11 06:50:37+00:00,4528,2026-03-06 04:43:18+00:00,0,Python,7328,0,1263,1227,0.9715
6,kamranahmedse/developer-roadmap,350262,2017-03-15 13:45:52+00:00,3277,2026-03-06 05:17:35+00:00,0,TypeScript,4073,1,3085,3054,0.9900
7,jwasham/coding-interview-university,337621,2016-06-06 02:34:12+00:00,3560,2026-03-06 03:23:56+00:00,0,None,415,0,525,454,0.8648
8,donnemartin/system-design-primer,337428,2017-02-26 16:15:28+00:00,3294,2026-03-06 05:08:48+00:00,0,Python,203,0,360,110,0.3056
9,vinta/awesome-python,285783,2014-06-27 21:00:06+00:00,4269,2026-03-06 05:11:17+00:00,0,Python,641,0,0,0,0.0000


## 6. Salvar dados

In [6]:
os.makedirs("data", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"data/repos_lab01_s01{timestamp}"

df.to_json(f"{filename}.json", orient="records", indent=2, force_ascii=False, date_format="iso")
print(f"Salvo: {filename}.json")

df.to_csv(f"{filename}.csv", index=False)
print(f"Salvo: {filename}.csv")

Salvo: data/repos_lab01_s0120260306_021927.json
Salvo: data/repos_lab01_s0120260306_021927.csv
